In [17]:
!pip install pandas 
!pip install numpy

In [4]:
df=pd.read_csv("realistic_nmap_training_dataset.csv")

In [5]:
df.head()

,Port_no,Service,State,Vulnerabilities,Tools,Next_steps,Commands
0,445,microsoft-ds,open,CVE-2017-0144|Weak Credentials|SMBv1 Enabled,metasploit|smbclient|enum4linux,Exploit SMB vulnerability,crackmapexec smb <target>|enum4linux <target>|...
1,27017,mongodb,closed,Weak Authentication|Unauthenticated MongoDB,metasploit|mongo,Enumerate databases|Check authentication settings,nmap --script mongodb-info -p 27017 <target>
2,22,ssh,closed,CVE-2018-15473,nmap,Check SSH version|Enumerate supported algorithms,nmap -sV -p 22 <target>
3,3306,mysql,filtered,CVE-2012-2122,sqlmap,Test default credentials|Enumerate databases,nmap --script mysql-info -p 3306 <target>|mysq...
4,443,https,filtered,Heartbleed,testssl,Scan SSL vulnerabilities|Check TLS configuration,testssl.sh <target>|sslscan <target>


In [6]:
df.tail()

,Port_no,Service,State,Vulnerabilities,Tools,Next_steps,Commands
995,80,http,open,CVE-2021-41773|Directory Listing|XSS,burpsuite|gobuster,Enumerate directories|Test login forms|Check w...,sqlmap -u http://<target>/login.php?id=1
996,443,https,closed,Weak TLS|Heartbleed|SSL Misconfiguration,nikto|sslscan|testssl,Check TLS configuration,sslscan <target>|testssl.sh <target>
997,443,https,closed,SSL Misconfiguration|Heartbleed,sslscan|nikto|testssl,Check TLS configuration|Scan SSL vulnerabilities,sslscan <target>
998,445,microsoft-ds,closed,Weak Credentials|SMBv1 Enabled,metasploit,Check SMB version,crackmapexec smb <target>|enum4linux <target>
999,27017,mongodb,open,Unauthenticated MongoDB|Weak Authentication,mongo|nmap,Enumerate databases|Check authentication settings,mongo --host <target>|nmap --script mongodb-in...


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Port_no          1000 non-null   int64 
 1   Service          1000 non-null   object
 2   State            1000 non-null   object
 3   Vulnerabilities  1000 non-null   object
 4   Tools            1000 non-null   object
 5   Next_steps       1000 non-null   object
 6   Commands         1000 non-null   object
dtypes: int64(1), object(6)
memory usage: 54.8+ KB


In [9]:
df.describe()

,Port_no
count,1000.000000
mean,4275.835000
std,8081.658128
min,21.000000
25%,80.000000
50%,445.000000
75%,3389.000000
max,27017.000000


In [11]:
df["Service"].unique()

array(['microsoft-ds', 'mongodb', 'ssh', 'mysql', 'https', 'snmp', 'rdp',
       'redis', 'http', 'ftp'], dtype=object)

In [12]:
df["Service"].nunique()

10

In [14]:
df["State"].unique()

array(['open', 'closed', 'filtered'], dtype=object)

In [15]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

# =========================
# LOAD DATASET
# =========================

df = pd.read_csv("realistic_nmap_training_dataset.csv")

# =========================
# INPUT FEATURES
# =========================

X = df[["Port_no", "Service", "State"]]

# =========================
# CONVERT PIPE VALUES TO LIST
# =========================

df["Vulnerabilities"] = df["Vulnerabilities"].apply(lambda x: x.split("|"))
df["Tools"] = df["Tools"].apply(lambda x: x.split("|"))
df["Next_steps"] = df["Next_steps"].apply(lambda x: x.split("|"))
df["Commands"] = df["Commands"].apply(lambda x: x.split("|"))

# =========================
# MULTI LABEL ENCODING
# =========================

vuln_mlb = MultiLabelBinarizer()
tool_mlb = MultiLabelBinarizer()
step_mlb = MultiLabelBinarizer()
command_mlb = MultiLabelBinarizer()

Y_vuln = vuln_mlb.fit_transform(df["Vulnerabilities"])
Y_tool = tool_mlb.fit_transform(df["Tools"])
Y_step = step_mlb.fit_transform(df["Next_steps"])
Y_command = command_mlb.fit_transform(df["Commands"])

# =========================
# MERGE OUTPUTS
# =========================

Y = np.concatenate([
    Y_vuln,
    Y_tool,
    Y_step,
    Y_command
], axis=1)

# =========================
# TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)

# =========================
# PREPROCESSING
# =========================

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Service", "State"]),
        ("num", "passthrough", ["Port_no"])
    ]
)

# =========================
# RANDOM FOREST MODEL
# =========================

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", MultiOutputClassifier(rf_model))
])

# =========================
# TRAIN MODEL
# =========================

print("Training model...")

model.fit(X_train, y_train)

print("Training completed!")

# =========================
# SAVE MODEL
# =========================

joblib.dump(model, "nmap_prediction_model.pkl")

joblib.dump(vuln_mlb, "vuln_encoder.pkl")
joblib.dump(tool_mlb, "tool_encoder.pkl")
joblib.dump(step_mlb, "step_encoder.pkl")
joblib.dump(command_mlb, "command_encoder.pkl")

print("Model saved successfully!")

# =========================
# TEST PREDICTION
# =========================

sample = pd.DataFrame([
    {
        "Port_no": 445,
        "Service": "microsoft-ds",
        "State": "open"
    }
])

prediction = model.predict(sample)

# =========================
# SPLIT PREDICTIONS
# =========================

vuln_end = len(vuln_mlb.classes_)
tool_end = vuln_end + len(tool_mlb.classes_)
step_end = tool_end + len(step_mlb.classes_)

pred_vuln = prediction[:, :vuln_end]
pred_tool = prediction[:, vuln_end:tool_end]
pred_step = prediction[:, tool_end:step_end]
pred_command = prediction[:, step_end:]

# =========================
# DECODE OUTPUT
# =========================

vulnerabilities = vuln_mlb.inverse_transform(pred_vuln)
tools = tool_mlb.inverse_transform(pred_tool)
steps = step_mlb.inverse_transform(pred_step)
commands = command_mlb.inverse_transform(pred_command)

# =========================
# OUTPUT
# =========================

print("\nPredicted Vulnerabilities:")
print(vulnerabilities)

print("\nPredicted Tools:")
print(tools)

print("\nPredicted Next Steps:")
print(steps)

print("\nPredicted Commands:")
print(commands)

Training model...
Training completed!
Model saved successfully!

Predicted Vulnerabilities:
[('CVE-2017-0144', 'SMBv1 Enabled', 'Weak Credentials')]

Predicted Tools:
[('crackmapexec', 'enum4linux', 'metasploit', 'smbclient')]

Predicted Next Steps:
[('Check SMB version', 'Enumerate SMB shares', 'Exploit SMB vulnerability')]

Predicted Commands:
[('crackmapexec smb <target>', 'enum4linux <target>', 'smbclient -L //<target>')]


In [19]:
df["Port_no"].unique()

array([  445, 27017,    22,  3306,   443,   161,  3389,  6379,    80,
          21])

In [18]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

In [19]:
df = pd.read_csv("realistic_nmap_training_dataset.csv")

In [20]:
X = df[["Port_no", "Service", "State"]]

(X.head())

,Port_no,Service,State
0,445,microsoft-ds,open
1,27017,mongodb,closed
2,22,ssh,closed
3,3306,mysql,filtered
4,443,https,filtered


In [21]:
df["Vulnerabilities"] = df["Vulnerabilities"].apply(lambda x: x.split("|"))
df["Tools"] = df["Tools"].apply(lambda x: x.split("|"))
df["Next_steps"] = df["Next_steps"].apply(lambda x: x.split("|"))
df["Commands"] = df["Commands"].apply(lambda x: x.split("|"))

In [22]:
vuln_mlb = MultiLabelBinarizer()
tool_mlb = MultiLabelBinarizer()
step_mlb = MultiLabelBinarizer()
command_mlb = MultiLabelBinarizer()

Y_vuln = vuln_mlb.fit_transform(df["Vulnerabilities"])
Y_tool = tool_mlb.fit_transform(df["Tools"])
Y_step = step_mlb.fit_transform(df["Next_steps"])
Y_command = command_mlb.fit_transform(df["Commands"])

In [23]:
Y = np.concatenate([
    Y_vuln,
    Y_tool,
    Y_step,
    Y_command
], axis=1)

print(Y.shape)

(1000, 92)


In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Service", "State"]),
        ("num", "passthrough", ["Port_no"])
    ]
)

In [26]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", MultiOutputClassifier(rf_model))
])

In [27]:
print("Training model...")

model.fit(X_train, y_train)

print("Training completed!")

Training model...
Training completed!


In [28]:
joblib.dump(model, "nmap_prediction_model.pkl")

joblib.dump(vuln_mlb, "vuln_encoder.pkl")
joblib.dump(tool_mlb, "tool_encoder.pkl")
joblib.dump(step_mlb, "step_encoder.pkl")
joblib.dump(command_mlb, "command_encoder.pkl")

print("Model saved successfully!")

Model saved successfully!


In [29]:
sample = pd.DataFrame([
    {
        "Port_no": 445,
        "Service": "microsoft-ds",
        "State": "open"
    }
])

prediction = model.predict(sample)

In [30]:
vuln_end = len(vuln_mlb.classes_)
tool_end = vuln_end + len(tool_mlb.classes_)
step_end = tool_end + len(step_mlb.classes_)

pred_vuln = prediction[:, :vuln_end]
pred_tool = prediction[:, vuln_end:tool_end]
pred_step = prediction[:, tool_end:step_end]
pred_command = prediction[:, step_end:]

In [31]:
vulnerabilities = vuln_mlb.inverse_transform(pred_vuln)
tools = tool_mlb.inverse_transform(pred_tool)
steps = step_mlb.inverse_transform(pred_step)
commands = command_mlb.inverse_transform(pred_command)

In [32]:
print("\nPredicted Vulnerabilities:")
print(vulnerabilities)

print("\nPredicted Tools:")
print(tools)

print("\nPredicted Next Steps:")
print(steps)

print("\nPredicted Commands:")
print(commands)


Predicted Vulnerabilities:
[('CVE-2017-0144', 'SMBv1 Enabled', 'Weak Credentials')]

Predicted Tools:
[('crackmapexec', 'enum4linux', 'metasploit', 'smbclient')]

Predicted Next Steps:
[('Check SMB version', 'Enumerate SMB shares', 'Exploit SMB vulnerability')]

Predicted Commands:
[('crackmapexec smb <target>', 'enum4linux <target>', 'smbclient -L //<target>')]
